## An example of LLM annotation on BANKING

#### Step 1. Generating Few-shot Examples

define prompt template for generation

In [ ]:
prompt = """ You are required to produce 100 English examples with labels for the task of intent classification on the BANKING77 dataset. These samples will be used as prompt examples for the GPT model. BANKING77 dataset is used in intent-analysis experiments and this dataset contains online banking queries labeled with respect to the intent. The task is to classify a banking query as one of the following 77 types: {}. For example, 100 of the unlabeled samples in BANKING77 dataset are as follows: {} The format of each example should be: query || label. """
prompt

sample unlabeled data

In [ ]:
import pandas as pd
task = "banking"
df = pd.read_csv(f'data/{task}/train.csv', header=None)
shuffled_df = df.sample(frac=1, random_state=45).reset_index(drop=True)

template = """["text": "{}"]"""
template = "[{}]"
templates = []
questions = shuffled_df[1].to_list()
for i in range(100):
    templates.append(template.format(questions[i]))
examples = ' '.join(templates)
examples

final prompt

In [ ]:
BANKING_CLS = ['pending card payment', 'card payment not recognised', 'reverted card payment', 'visa or mastercard', 'fiat currency support', 'verify source of funds', 'refund not showing up', 'pending top up', 'getting virtual card', 'compromised card', 'supported cards and currencies', 'lost or stolen phone', 'cash withdrawal charge', 'edit personal details', 'wrong exchange rate for cash withdrawal', 'exchange charge', 'declined card payment', 'card arrival', 'automatic top up', 'top up by card charge', 'top up reverted', 'topping up by card', 'terminate account', 'top up failed', 'unable to verify identity', 'pending cash withdrawal', 'top up by cash or cheque', 'disposable card limits', 'balance not updated after cheque or cash deposit', 'card swallowed', 'balance not updated after bank transfer', 'receiving money', 'card linking', 'activate my card', 'exchange rate', 'verify top up', 'why verify identity', 'pending transfer', 'card delivery estimate', 'direct debit payment not recognised', 'get disposable virtual card', 'card payment wrong exchange rate', 'transfer fee charged', 'passcode forgotten', 'top up by bank transfer charge', 'order physical card', 'declined cash withdrawal', 'transfer timing', 'extra charge on statement', 'cash withdrawal not recognised', 'declined transfer', 'beneficiary not allowed', 'failed transfer', 'transaction charged twice', 'country support', 'lost or stolen card', 'pin blocked', 'card acceptance', 'wrong amount of cash received', 'card payment fee charged', 'top up limits', 'getting spare card', 'transfer into account', 'card about to expire', 'virtual card not working', 'card not working', 'apple pay or google pay', 'transfer not received by recipient', 'get physical card', 'request refund', 'contactless not working', 'atm support', 'verify my identity', 'exchange via app', 'change pin', 'age limit', 'cancel transfer']
prompt_final = prompt.format('; '.join(BANKING_CLS), examples)
prompt_final

Then you can gather the generated texts into demo.csv, along with their embeddings and labels, see the example in "data\banking\demo.csv" \
After that, you can calculate the embedding similarity matrices for few-shot sample selection, see examples in "data\banking\similarity_train.npy"

#### Step 2. Calling LLM

task definition and parameter setting

In [ ]:
task, split, ptype = "banking", "train", "add"

define prompt template for annotation

In [ ]:
BANKING_SENTENCE = "Given a query: '{}'"
BANKING_SYSTEM = "You are a helpful assistant for the task of intent classification on the BANKING77 dataset. You reply with brief, to-the-point answers with no elaboration as truthfully as possible. BANKING77 dataset contains 13083 online banking queries labeled with 77 types of intents: {}. Your task is to classify the given queries as one of the 77 given intents. "
BANKING_USER_DEMO = " What is the intent of the given query ? Please identify the query with one of the 77 intents mentioned above. "
BANKING_USER_DICT = {
    "single":" What is the intent of the given query ? Please identify the query into one of the 77 intents mentioned above. Do not answer any other classses or anything else.",
    "add": " What is the intent of the given query ? Please identify the query into one of the 77 intents mentioned above. \
    If you are unsure about your answer, please include other potential choices. Separate your choices by '||'. Do not answer any other classses or anything else.",
    "all": " What is the intent of the given query ? Please identify the query with all possible choices of the 77 intents mentioned above. \
    Separate your choices by '||'. Do not answer any other classses or anything else.",
}

generating LLM batch request

In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from prompts import *

def gen_jsonl(task, split, ptype):    
    df = pd.read_csv(f'data/{task}/{split}.csv', header=None)
    all_texts = df[1].to_list()
    num_texts = len(all_texts)
    n_icl = 10

    df_demo = pd.read_csv(f'data/{task}/demo.csv')
    demo_sim = np.load(f'data/{task}/similarity_{split}.npy')
    demo_texts = df_demo['text'].tolist()
    demo_annos = df_demo['anno'].tolist()

    class_names = BANKING_CLS
    sentence_prompt = BANKING_SENTENCE
    system_prompt = BANKING_SYSTEM
    user_prompt_demo = BANKING_USER_DEMO
    user_prompt = BANKING_USER_DICT[ptype]
    
    name2idx = {}
    for i, n in enumerate(class_names):
        name2idx[n.lower()] = i
    
    data = []
    for i in tqdm(range(num_texts)):
        temp_messages = [{"role": "system", "content": system_prompt.format('; '.join(class_names))}]
        for j in range(n_icl):
            temp_sentence = demo_texts[demo_sim[i][j]].replace("\n", " ").replace('"', '\"')
            temp_anno = demo_annos[demo_sim[i][j]]
            temp_messages.append({"role": "user", "content": sentence_prompt.format(temp_sentence)})
            temp_messages.append({"role": "user", "content": user_prompt_demo})
            temp_messages.append({"role": "assistant", "content": temp_anno})
        
        sentence = all_texts[i]
        sentence = sentence.replace("\n", " ").replace('"', '\"')
        temp_messages.append({"role": "user", "content": sentence_prompt.format(sentence)})
        temp_messages.append({"role": "user", "content": user_prompt})

        data.append(
            {
                "custom_id": f"request-{i+1}",
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-3.5-turbo-0125",
                    "messages": temp_messages,
                    "max_tokens": 1000,
                    "n": 1,
                    "temperature": 0.3,
                }
            }
        )
    path = f'llm_requests/{task}_{split}_{ptype}.jsonl'
    print('save to:', path)
    with open(path, 'w') as file:
        for item in data:
            file.write(json.dumps(item) + '\n')

gen_jsonl(task, split, ptype)

upload batch request and start working

In [ ]:
import openai
from openai import OpenAI

openai.api_key = 'sk-xxx'  # your api here
client = OpenAI()

batch_input_file = client.files.create(
  file=open(f'llm_requests/{task}_{split}_{ptype}.jsonl', "rb"),
  purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
      "description": "nightly eval job"
    }
)

check status

In [ ]:
from openai import OpenAI
client = OpenAI()
batch = client.batches.retrieve(batch.id)
batch.status, batch.request_counts

get output when completed

In [ ]:
from openai import OpenAI
client = OpenAI()
file_response = client.files.content(batch.output_file_id)
file_response_lst = file_response.text.split('\n')[:-1]
output_lst = []
for frl in file_response_lst:
    output_lst.append(eval(frl.replace('null', 'None')))
# ranking
sorted_output_lst = sorted(output_lst, key=lambda x: int(x['custom_id'][8:]))

import json
with open(f'llm_outputs/{task}_{split}_{ptype}.jsonl', 'w') as file:
    for item in sorted_output_lst:
        file.write(json.dumps(item) + '\n')

#### Step 3. Evaluation

load functions

In [ ]:
import numpy as np

def post_process(output: str):
    if len(output) == 0:
        return ''
    if output[-1] == '.':
        output = output[:-1]
    if output[-1] == ',':
        output = output[:-1]
    output = output.replace('\n', ' ')
    return output.strip().lower()

def get_result(text: str, class_names: str):
    t_lst = []
    for cn in class_names:
        count = text.lower().count(cn.lower())
        t_lst.append(count)
    if sum(t_lst) == 0:
        return 'none_class'
    else:
        indices = np.argsort(t_lst)[-2:]
        idx0 = indices[0]
        idx1 = indices[1]
        if t_lst[idx0] == t_lst[idx1] and (class_names[idx0] in class_names[idx1] or class_names[idx1] in class_names[idx0]):
            res = class_names[idx0] if len(class_names[idx0]) > len(class_names[idx1]) else class_names[idx1]
            return res.lower()
        return class_names[np.argmax(t_lst)].lower()

load LLM's outputs and ground-truth labels

In [ ]:
import json
import pandas as pd

with open(f'llm_outputs/{task}_{split}_{ptype}.jsonl', 'r') as infile:
    data = [json.loads(line) for line in infile]

df_true = pd.read_csv(f'data/{task}/{split}.csv', header=None)
t_labels = df_true[0].to_list()
texts = df_true[1].to_list()

In [ ]:
class_names = BANKING_CLS
name2idx = {}
for i, n in enumerate(class_names):
    name2idx[n.lower()] = i
ds2num = {'trec': 6, 'ma': 5, 'dbpedia': 14, 'agnews': 4, 'rct': 5, 'banking': 77}

def eval_f1(acc, label_num):
    alpha = 1 - acc
    beta = (ds2num[task] - label_num) / (ds2num[task] - 1)
    F1_score = 2 * (1 - alpha) * beta / (1 - alpha + beta)
    print('F1:', round(F1_score, 4))

calculating metrics

In [ ]:
import numpy as np
gpt_labels = []
wrong_idx = []
if ptype == 'single':
    for i in range(len(data)):
        di = data[i]['response']['body']['choices'][0]['message']['content']
        di = post_process(di)
        di = get_result(di, class_names)
        gpt_label = name2idx[di] if di in name2idx.keys() else -1
        gpt_labels.append(gpt_label)
        if t_labels[i] != gpt_label:
            wrong_idx.append(i)
    acc = (len(gpt_labels)-len(wrong_idx)) / len(gpt_labels)
    print(f'Annotation Acc: {acc}')
    eval_f1(acc, 1.0)
else:
    for i in range(len(data)):
        di = data[i]['response']['body']['choices'][0]['message']['content']
        di = post_process(di)
        di_lst = [get_result(post_process(d), class_names) if len(d) else '' for d in di.split('||')]
        gpt_label = [name2idx[dl] if dl in name2idx.keys() else -1 for dl in di_lst]
        gpt_label = list(dict.fromkeys(gpt_label))
        gpt_labels.append(gpt_label)
        if t_labels[i] not in gpt_label:
            wrong_idx.append(i)
    acc = (len(gpt_labels)-len(wrong_idx)) / len(gpt_labels)
    print(f'Annotation Acc: {acc}')
    p_num = []
    for pl in gpt_labels:
        while -1 in pl:
            pl.remove(-1)
        if len(pl) == 0:
            p_num.append(1)
        else:
            p_num.append(len(pl))
    label_num = np.array(p_num).mean()
    print('Avg labels:', label_num)
    eval_f1(acc, label_num)

saving to csv file for distillation

In [ ]:
df_p = pd.DataFrame({
    'anno': gpt_labels,
    'text': texts,
})
df_p.to_csv(f'data/{task}/{split}_{ptype}.csv', index=False, header=None)
print('Save to:', f'data/{task}/{split}_{ptype}.csv')